# 04 · Comparison & TDC Leaderboard
전 모델(RF / ECFP-MLP / GCN / GIN) × 전 엔드포인트 성능 테이블, 막대·레이더 차트, **TDC 리더보드 공개 점수와 나란히 비교(차이/순위 추정)**, Tox21 레포 연계 서술.

In [ ]:
# --- Colab 자가치유 셋업 (로컬/CI에서는 자동 스킵) ---
import os, sys, subprocess
try:
    import src  # repo 루트에서 실행 중이면 바로 import
except ModuleNotFoundError:
    if not os.path.exists("admet-property-predictor"):
        subprocess.run(["git", "clone",
            "https://github.com/zqvo04/admet-property-predictor.git"], check=False)
    os.chdir("admet-property-predictor")
    subprocess.run(["bash", "setup_colab.sh"], check=False)
    import src
print("src", src.__version__, "| cwd", os.getcwd())

In [ ]:
# === CONFIG (Colab에서 값만 키우면 전체 재현) ===
# 검증용 기본값은 작게 설정 — 실제 학습 시 EPOCHS↑, 엔드포인트 추가.
import os
TDC_PATH = "data/"
EPOCHS   = int(os.environ.get("ADMET_EPOCHS", 2))   # Colab 실전: 100
REG_EP   = ["Caco2"]          # 회귀 엔드포인트 (확장: Solubility, Lipophilicity, ...)
CLS_EP   = ["hERG"]           # 분류 엔드포인트 (확장: BBBP, CYP3A4_Inhibition, ...)
ENDPOINTS_USED = REG_EP + CLS_EP
SEED = 1
print("endpoints:", ENDPOINTS_USED, "| epochs:", EPOCHS)

## 결과 취합
여기서는 02/03 노트북의 산출(또는 5-seed 평균)을 모은 표를 사용. 데모용으로 단일-seed 빠른 재학습 결과를 채운다.

In [ ]:
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as GDataLoader
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from src.data import load_tdc_endpoint, build_multitask, ENDPOINTS
from src.featurizers import ecfp_featurize, smiles_list_to_graphs
from src.models import build_model
from src.train import Trainer, TrainConfig, ECFPDataset, graph_dataset
from src.evaluate import evaluate_multitask, regression_metrics, classification_metrics
np.random.seed(SEED); torch.manual_seed(SEED)
data = {ep: load_tdc_endpoint(ep, seed=SEED, tdc_path=TDC_PATH) for ep in ENDPOINTS_USED}
task_names=list(data.keys()); task_types=[data[n].task_type for n in task_names]
type_vec=[0 if t=='reg' else 1 for t in task_types]

In [ ]:
def official(ep, m): return m.get(ENDPOINTS[ep]['metric'])
scores = {ep: {} for ep in ENDPOINTS_USED}
# RF
for ep,d in data.items():
    Xtr=ecfp_featurize(d.smiles('train')); Xte=ecfp_featurize(d.smiles('test'))
    if d.task_type=='reg':
        rf=RandomForestRegressor(150,n_jobs=-1,random_state=SEED).fit(Xtr,d.targets('train'))
        scores[ep]['RF']=official(ep, regression_metrics(d.targets('test'), rf.predict(Xte)))
    else:
        rf=RandomForestClassifier(150,n_jobs=-1,random_state=SEED).fit(Xtr,d.targets('train'))
        scores[ep]['RF']=official(ep, classification_metrics(d.targets('test'), rf.predict_proba(Xte)[:,1]))
scores

In [ ]:
# ECFP-MLP / GCN / GIN (멀티태스크)
mt_tr=build_multitask(data,'train'); mt_te=build_multitask(data,'test')
def ecfp_loaders():
    tl=DataLoader(ECFPDataset(ecfp_featurize(mt_tr.smiles),mt_tr.Y,mt_tr.mask),batch_size=64,shuffle=True)
    tel=DataLoader(ECFPDataset(ecfp_featurize(mt_te.smiles),mt_te.Y,mt_te.mask),batch_size=64)
    return tl,tel
def graph_loaders():
    g_tr,m_tr=smiles_list_to_graphs(mt_tr.smiles); g_te,m_te=smiles_list_to_graphs(mt_te.smiles)
    tl=GDataLoader(graph_dataset(g_tr,mt_tr.Y[m_tr],mt_tr.mask[m_tr]),batch_size=64,shuffle=True)
    tel=GDataLoader(graph_dataset(g_te,mt_te.Y[m_te],mt_te.mask[m_te]),batch_size=64)
    return tl,tel,m_te
configs=[('ECFP-MLP','ecfp'),('GCN','gcn'),('GIN','gin')]
for label,kind in configs:
    if kind=='ecfp':
        tl,tel=ecfp_loaders(); Yte,Mte=mt_te.Y,mt_te.mask
    else:
        tl,tel,m_te=graph_loaders(); Yte,Mte=mt_te.Y[m_te],mt_te.mask[m_te]
    model=build_model(kind,num_tasks=len(task_names))
    tr=Trainer(model,task_types=type_vec,cfg=TrainConfig(epochs=EPOCHS,ckpt_dir='checkpoints/',run_name=label,verbose=False))
    tr.fit(tl)
    r=evaluate_multitask(tr.predict(tel),Yte,Mte,task_names,task_types,mt_te.scalers)
    for ep in ENDPOINTS_USED: scores[ep][label]=official(ep,r[ep])
score_df=pd.DataFrame(scores).T; score_df

## 성능 막대 차트 (엔드포인트별 모델 비교)

In [ ]:
ax=score_df.plot(kind='bar', figsize=(8,4))
ax.set_ylabel('official metric value'); ax.set_title('Model comparison per endpoint')
plt.xticks(rotation=0); plt.tight_layout()
import os; os.makedirs('results/figures',exist_ok=True)
plt.savefig('results/figures/model_comparison.png',dpi=120); plt.show()

## TDC 리더보드 대비 비교
공개 리더보드 best 점수(예시 값, 실제 제출 시 갱신)와 우리 결과를 나란히 두고 차이/순위 추정.

In [ ]:
# TDC 공개 리더보드 best (참고용 스냅샷; tdcommons.ai 기준 근사치)
TDC_LEADERBOARD = {
    'Caco2': {'metric':'mae','best':0.276,'best_method':'MapLight'},
    'hERG':  {'metric':'roc-auc','best':0.880,'best_method':'BaseBoosting'},
    'Solubility': {'metric':'mae','best':0.761,'best_method':'SimGCN'},
    'Lipophilicity': {'metric':'mae','best':0.467,'best_method':'CFA'},
    'BBBP': {'metric':'roc-auc','best':0.916,'best_method':'ContextPred'},
    'Bioavailability': {'metric':'roc-auc','best':0.748,'best_method':'RDKit2D'},
    'CYP3A4_Inhibition': {'metric':'pr-auc','best':0.876,'best_method':'XGBoost'},
}
rows=[]
for ep in ENDPOINTS_USED:
    ours=score_df.loc[ep].max()  # 우리 best 모델
    best_model=score_df.loc[ep].idxmax()
    lb=TDC_LEADERBOARD.get(ep,{})
    metric=ENDPOINTS[ep]['metric']
    higher_better = metric in ('roc-auc','pr-auc','spearman','pearson')
    gap = (ours-lb['best']) if lb else np.nan
    rows.append({'endpoint':ep,'metric':metric,'our_best':round(float(ours),3),
                 'our_model':best_model,'tdc_best':lb.get('best'),
                 'tdc_method':lb.get('best_method'),
                 'gap(ours-tdc)':round(float(gap),3) if lb else None,
                 'higher_better':higher_better})
cmp_df=pd.DataFrame(rows); cmp_df

## 5-seed 공식 제출 프로토콜 (리더보드 제출용 스니펫)
리더보드 제출 시에는 아래처럼 seed [1..5] 평균±std를 `group.evaluate_many`로 산출.

In [ ]:
# 제출용 (실행은 Colab에서; 시간 소요로 데모는 주석)
snippet = '''
from tdc.benchmark_group import admet_group
group = admet_group(path=TDC_PATH)
preds_list = []
for seed in [1,2,3,4,5]:
    # train model with this seed split, predict on fixed test
    preds_list.append({ENDPOINTS[ep]["tdc_name"]: y_pred_test})
print(group.evaluate_many(preds_list))  # {name: [mean, std]}
'''
print(snippet)

## 신약 파이프라인 시리즈 연계
- **1/4 Tox21** (분류, ECFP vs GraphConv): 멀티레이블 독성 — 마스킹 손실/공통 Trainer 도입.
- **2/4 ADMET (본 레포)**: 회귀+분류 혼합 멀티태스크, TDC 표준 split·공식 지표·리더보드 비교.
- **3/4 분자 생성** (예정): 우수 ADMET 프로파일 분자 생성(VAE/Flow/Diffusion).
- **4/4 결합 예측** (예정): 단백질-리간드 결합/도킹 스코어 예측.

동일 모듈 구조(data/featurizers/models/losses/train/evaluate)와 공통 Trainer로 한 시리즈로 읽히게 유지.